In [1]:
import os
import sys
os.chdir('/zhome/71/c/146676/texture_tomography')
sys.path.append('/zhome/71/c/146676/texture_tomography/package/odf_mumott')
import numpy as np

from package.cil_addons.texture_tomography.operators.pfo_and_projection_batched_opencl import PFO_OPENCL_BATCHED
from package.cil_addons.texture_tomography.operators.operator_memory_model import OperatorMemoryModel
from package.cil_addons.texture_tomography.operators.memory_tracker import MemoryCounter
from package.cil_addons.texture_tomography.optimization.fista_opencl import FISTAOpenCL
from package.cil_addons.texture_tomography.optimization.fista_memory_model import FISTAMemoryModel

import matplotlib.pyplot as plt

import yaml

INFO:Setting the number of threads to 8. If your physical cores are fewer than this number, you may want to use numba.set_num_threads(n), and os.environ["OPENBLAS_NUM_THREADS"] = f"{n}" to set the number of threads to the number of physical cores n.
INFO:Setting numba log level to WARNING.


In [2]:
config_path = "configs/aluminum_config_medium.yaml"
with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

N_theta = 100
two_thetas = np.linspace(0.01,0.6,N_theta)

op = PFO_OPENCL_BATCHED(
        cfg = cfg,
        two_thetas = two_thetas,
        verbose=False)
op.set_pf_batch_max_gb(4.0)

mem = MemoryCounter()
memory_model = OperatorMemoryModel(op, mem)
memory_model.mem.report(show_peak=True)
memory_model.mem.report_allocations()
memory_model.model_direct()
memory_model.mem.report(show_peak=True)
memory_model.mem.report_allocations()
memory_model.model_adjoint()
memory_model.mem.report(show_peak=True)
memory_model.mem.report_allocations()

=== MemoryCounter Report ===
Current : 3.719 GB
Peak    : 3.719 GB
Live allocations : 8

=== Peak Allocation Breakdown ===
Name                                        Size (MB)
-------------------------------------------------------
_out_sub[0]                                  3048.706
_coeffs_t_gpu[0]                              735.077
pf_coords[0]                                   24.719
pf_grid_inv[0]                                  0.045
full_idx[0]                                     0.021
pf_sym_ops[0]                                   0.001
pf_h[0]                                         0.000
pf_intensity[0]                                 0.000
-------------------------------------------------------
TOTAL @ PEAK                                 3808.569


=== Live Allocation Breakdown ===
Name                                        Size (MB)
-------------------------------------------------------
_out_sub[0]                                  3048.706
_coeffs_t_gpu[0]         

In [3]:
mem = MemoryCounter()

op_model = OperatorMemoryModel(op, mem)

fista = FISTAOpenCL(
    operator=op,
    prox_kind="nonneg_tv",
    lam=0.0,
    tau=1e-3
)

fista_mem = FISTAMemoryModel(fista, mem, op_model)

x_shape  = (op.Nx, op.Nx, op.K_sum)
Ax_shape = (op.N_rot, op.Nx, op.N_chi * op.N_theta)

fista_mem.model_run(x_shape, Ax_shape, niter=1)

mem.report()
mem.report_peak_allocations(min_mb=50)


=== MemoryCounter Report ===
Current : 24.736 GB
Peak    : 49.566 GB
Live allocations : 21

=== Peak Allocation Breakdown ===
Name                                        Size (MB)
-------------------------------------------------------
fista.Ax                                    10162.354
fista.r                                     10162.354
yin_gpu                                     10162.354
B_gpu_batch[m0,b0]                           4078.674
BT_gpu_batch[m0,b0]                          4078.674
_out_sub[0]                                  3048.706
data_gpu_sub[m0]                             3048.706
pf_basis_batch[m0,b0]                        1631.470
_coeffs_t_gpu[0]                              735.077
_coeffs_gpu_full_sino                         735.077
_x_full_gpu                                   735.077
xin_gpu[m0]                                   735.077
x_batch[m0,b0]                                111.786
fista.x                                       108.791
fista.y 